In [1]:
pip install folium


   -------------------- ------------------- 1/2 [folium]
   -------------------- ------------------- 1/2 [folium]
   -------------------- ------------------- 1/2 [folium]
   -------------------- ------------------- 1/2 [folium]
   ---------------------------------------- 2/2 [folium]

Note: you may need to restart the kernel to use updated packages.


In [14]:
pip install geopy


   ---------------------------------------- 0/2 [geographiclib]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   ---------------------------------------- 2/2 [geopy]

Note: you may need to restart the kernel to use updated packages.


In [55]:
pip install geopandas

   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
   ---------------------------------------  23.6/23.8 MB 115.0 MB/s eta 0:00:01
   ---------------------------------------- 23.8/23.8 MB 96.2 MB/s  0:00:00
   ---------------------------------------- 0.0/6.9 MB ? eta -:--:--
   ---------------------------------------- 6.9/6.9 MB 100.9 MB/s  0:00:00
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 91.4 MB/s  0:00:00

   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   -------------------

In [55]:
import json
import pandas as pd
import folium 
import math
from geopy.distance import geodesic
import geopandas as gpd
from shapely.geometry import shape
from scipy.spatial.distance import pdist #Permite calcular la Distancia entre puntos
from scipy.spatial import cKDTree #Recherche de plus proches voisins et Métriques de distance
import numpy as np

pd.set_option('display.max_colwidth', None)

## Funtion de charge general
def charger_data():
        
    pd.set_option('display.max_rows', 1000)
    data_arbres = pd.read_csv('../data/trees.csv')
    return data_arbres


In [2]:

data_arbres_details = charger_data()
data_arbres_details.index
data_arbres_details.dtypes

def coords_validas(gj):
    try:
        geom = gj.get('geometry', gj)
        coords = geom.get('coordinates')
        if coords is None:
            return False
        # aplanar y revisar
        def flat(c):
            if isinstance(c[0], (int, float)):
                yield c
            else:
                for x in c:
                    yield from flat(x)
        for pt in flat(coords):
            lon, lat = pt[0], pt[1]
            if not (-180 <= lon <= 180 and -90 <= lat <= 90):
                return False
        return True
    except Exception:
        return False
carta = folium.Map(location=[45.154, 5760], zoom_start=13)

features = []
for idx, row in data_arbres_details.iterrows():
    geojson_str = row['GeoJSON']

#Convertir str a Dic
    if geojson_str is None or (isinstance(geojson_str, float) and pd.isna(geojson_str)):
        continue
    geojson_data = json.loads(geojson_str) if isinstance(geojson_str, str) else geojson_str
    folium.GeoJson(
        geojson_data, #Contiene la geometria completa
        tooltip=f"Árbol {idx}",
        name=f"Elemente {idx}",
        style_function=lambda x: {
            "color": "green", 
            "weight": 2
        },
        marker=folium.CircleMarker(radius=5, color="green", fill=True)
    ).add_to(carta)

carta.save('mapa.html')   

    # for idx, row in data_arbres_details.iterrows():
    #     raw = row['GeoJSON']
    #     if raw is None or (isinstance(raw, float) and pd.isna(raw)):
    #         continue
    #     gj = json.loads(raw) if isinstance(raw, str) else raw
    #     t = gj.get('type')
    #     coords = gj.get('coordinates')
    #     print(idx, t, coords)

##Comprobacion que los datos son homogeneos
    # tipos = {}
    # for raw in data_arbres_details['GeoJSON']:
    #     if raw is None or (isinstance(raw, float) and pd.isna(raw)):
    #         tipos['NaN'] = tipos.get('NaN', 0) + 1
    #         continue
    #     gj = json.loads(raw) if isinstance(raw, str) else raw
    #     t = gj.get('type', 'SIN_TYPE')
    #     tipos[t] = tipos.get(t, 0) + 1
    # print(tipos)

# for idx, row in data_arbres_details.iterrows():
#     gj = json.loads(row['GeoJSON']) if isinstance(row['GeoJSON'], str) else row['GeoJSON']
# print(idx, gj.get('geometry', {}).get('type'), gj.get('geometry', {}).get('coordinates'))

        
# carte_arbres = fl.

<h1> intento con zoom </h1>

In [3]:
from folium.plugins import MarkerCluster

def distribution_geographique_zoom(data):
    
    # 1. Extraer coordenadas de una vez (rápido y seguro)
    lats, lons = [], []
    for raw in data['GeoJSON']:
        if raw is None or (isinstance(raw, float) and pd.isna(raw)):
            continue
        gj = json.loads(raw) if isinstance(raw, str) else raw
        lon, lat = gj['coordinates']
        lons.append(lon)
        lats.append(lat)
    
    if not lats:
        raise ValueError("No hay coordenadas válidas")
    
    centro = [sum(lats)/len(lats), sum(lons)/len(lons)]   # ✅ FUERA del for
    print(f"Centro: {centro} | Puntos: {len(lats)}")
    
    # 2. Mapa base
    m = folium.Map(location=centro, zoom_start=13, tiles="OpenStreetMap")
    
    # 3. Cluster
    cluster = MarkerCluster(
        name="Árboles",
        options={
            "maxClusterRadius": 50,
            "disableClusteringAtZoom": 18,
            "spiderfyOnMaxZoom": True,
        }
    ).add_to(m)
    
    # 4. Puntos con CircleMarker (ligero, sin FontAwesome)
    for idx, row in data.iterrows():
        raw = row['GeoJSON']
        if raw is None or (isinstance(raw, float) and pd.isna(raw)):
            continue
        gj = json.loads(raw) if isinstance(raw, str) else raw
        lon, lat = gj['coordinates']
        
        folium.CircleMarker(
            location=[lat, lon],
            radius=4,
            color="green",
            fill=True,
            fill_color="green",
            fill_opacity=0.7,
            tooltip=f"Árbol {idx}",
        ).add_to(cluster)
    
    folium.LayerControl().add_to(m)
    m.save("mapa_cluster.html")
    print("✅ mapa_cluster.html guardado")
    return m
data_arbres_details = charger_data()
distribution_geographique_zoom(data_arbres_details)

Centro: [45.1768308498065, 5.727216441477262] | Puntos: 31243
✅ mapa_cluster.html guardado


<h1>intento sin zoom</h1>

In [4]:
data_arbres_details = charger_data()
def distribution_geographique(data):
    # centro del mapa
    lats = []
    lons = []
    for raw in data['GeoJSON']:
        if raw is None or (isinstance(raw, float) and pd.isna(raw)):
            continue
        gj = json.loads(raw) if isinstance(raw, str) else raw
        lon, lat = gj['coordinates']
        lons.append(lon)
        lats.append(lat)
    
    m = folium.Map(location=[sum(lats)/len(lats), sum(lons)/len(lons)], zoom_start=13)
    
    for idx, row in data.iterrows():
        raw = row['GeoJSON']
        if raw is None or (isinstance(raw, float) and pd.isna(raw)):
            continue
        gj = json.loads(raw) if isinstance(raw, str) else raw
        lon, lat = gj['coordinates']
    
        folium.CircleMarker(
            location=[lat, lon],          # ⚠️ Folium usa [lat, lon], NO [lon, lat]
            radius=4,
            color="green",
            fill=True,
            fill_color="green",
            fill_opacity=0.7,
            tooltip=f"Árbol {idx}",
        ).add_to(m)

    m.save("mapa.html")
    print("✅ mapa_cluster.html guardado")
    return m

distribution_geographique(data_arbres_details)

✅ mapa_cluster.html guardado


<h1> Calculer la distance entre deux arbres de deux manières différentes</h1>
<h2> module geopy</h2>

In [5]:

## La geometría en formato GeoJSON [lon, lat]
## Extraer (lat, lon)
def extraer_lat_lon(raw):
    """Convierte el GeoJSON '[lon, lat]' en una tupla (lat, lon)."""
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return None
    gj = json.loads(raw) if isinstance(raw, str) else raw
    lon, lat = gj['coordinates']   # ⚠️ GeoJSON = [lon, lat]
    return (lat, lon)


def distance_geopy(coord1, coord2):
    """
    Calcula la distancia en metros entre dos puntos usando geopy.
    
    Parámetros:
        coord1, coord2 : tuplas (lat, lon) en grados decimales.
    
    Retorna:
        Distancia en metros (float).
    
    Nota: geodesic() usa el modelo WGS-84 (elipsoide),
          por eso es más preciso que Haversine (esfera perfecta).
    """
    # geodesic acepta (lat, lon) y devuelve un objeto Distance
    d = geodesic(coord1, coord2)
    # d = f"{d:.2f}"
    
    # .meters devuelve la distancia en metros (también hay .km, .miles...)
    return d.meters

# Funtion qui reçoit une liste contenant les GeoJson
# Des 2 arbres sélectionnés et renvoie la distance
def calculer_distance(liste_geojson):
    arbre1 = extraer_lat_lon(liste_geojson[0])
    arbre2 = extraer_lat_lon(liste_geojson[1])

    distance = distance_geopy(arbre1, arbre2)
    return distance


data_arbres_grenoble = charger_data()
data_arbres_grenoble.dtypes
data_arbres_grenoble = data_arbres_grenoble.sort_values('ANNEEDEPLANTATION')
data_arbres_grenoble.head(2)
arbres = data_arbres_grenoble['GeoJSON'].head(2).tolist()
distance_arbre = calculer_distance(arbres)
print(f"La distance entre les deux arbres indiqués est: {distance_arbre:.2f} m")


La distance entre les deux arbres indiqués est: 978.50 m


<h1>Distance entre deux arbres avec la formule de Haversine</h1>

In [6]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Calcula la distancia en metros entre dos puntos sobre la superficie
    de la Tierra usando la fórmula de Haversine.

    Parámetros:
        lat1, lon1 : latitud y longitud del primer punto (en grados)
        lat2, lon2 : latitud y longitud del segundo punto (en grados)

    Retorna:
        Distancia en metros (float)
    """
    # Radio medio de la Tierra en metros.
    # Se usa el radio medio (6371 km) porque la Tierra no es una esfera perfecta.
    R = 6371000.0

    # 1) Convertir las coordenadas de grados a radianes.
    #    Las funciones trigonométricas de Python trabajan en radianes.
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)   # Diferencia de latitudes
    dlambda = math.radians(lon2 - lon1) # Diferencia de longitudes

    # 2) Fórmula de Haversine:
    #    a = sin²(Δφ/2) + cos(φ1)·cos(φ2)·sin²(Δλ/2)
    a = (math.sin(dphi / 2) ** 2
         + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2)

    # 3) Ángulo central entre los dos puntos:
    #    c = 2 · atan2(√a, √(1−a))
    #    Se usa atan2 en lugar de asin para mayor estabilidad numérica
    #    cuando los puntos están muy cerca o en las antípodas.
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    # 4) Distancia = radio × ángulo central (en radianes)
    return R * c


# --- Ejemplo de uso ---
# Coordenadas de dos "árboles" (latitud, longitud)
# arbol1 = (40.4168, -3.7038)   # Madrid
# arbol2 = (41.3874, 2.1686)    # Barcelona

## La geometría en formato GeoJSON [lon, lat]
## Extraer (lat, lon)
def extraer_lat_lon(raw):
    """Convierte el GeoJSON '[lon, lat]' en una tupla (lat, lon)."""
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return None
    gj = json.loads(raw) if isinstance(raw, str) else raw
    lon, lat = gj['coordinates']   # ⚠️ GeoJSON = [lon, lat]
    return (lat, lon)

# Funtion qui reçoit une liste contenant les GeoJson
# Des 2 arbres sélectionnés et renvoie la distance
def calculer_distance(liste_geojson):
    arbre1 = extraer_lat_lon(liste_geojson[0])
    arbre2 = extraer_lat_lon(liste_geojson[1])
    print(arbre1, arbre2)
    distance = haversine(arbre1[0], arbre1[1], arbre2[0], arbre2[1])
    return distance
    
data_arbres_grenoble = charger_data()
data_arbres_grenoble = data_arbres_grenoble.sort_values('ANNEEDEPLANTATION')
data_arbres_grenoble.head(2)
arbres = data_arbres_grenoble['GeoJSON'].head(2).tolist()
distance_arbre = calculer_distance(arbres)

# distancia = haversine(arbol1[0], arbol1[1], arbol2[0], arbol2[1])
print(f"Distancia: {distance_arbre:.2f} m  ({distance_arbre/1000:.2f} km)")

(45.1922394744442, 5.72691643878808) (45.1980083607029, 5.71750935543133)
Distancia: 977.16 m  (0.98 km)


<h1>Quels sont les 10 arbres les plus proches au Campus numérique</h1>

In [7]:
## J'appelle la fonction de chargement pour créer una 
## Object de type DateFrame contenant les informations des arbres de Grenoble
data_arbres_proches = charger_data()

### Coordonnées du campus numérique obtenues via google map
## 45.184399713185094, 5.73136694086688
# J,assigne les Coordenadas de Le TOTEM, Grenoble
lon_totem = 5.73136694086688
lat_totem = 45.184399713185094



# Convertir GeoJSON en geometrías
df_arboles = data_arbres_proches.copy()

df_arboles["geometry"] = df_arboles["GeoJSON"].apply(
    lambda x: shape(json.loads(x)) if pd.notna(x) else None
)

# Crear GeoDataFrame
"""Le code EPSG:4326 correspond au système de coordonnées géographiques WGS 84 (World Geodetic System 1984).
Caractéristiques principales
• Type de coordonnées : Latitude et Longitude (2D).
• Unité de mesure : Degrés décimaux.
• Étendue : Mondiale (de -180° à 180° en longitude, de -90° à 90° en latitude).
• Utilisation principale : Référencement natif des récepteurs GPS, stockage de données géographiques brutes et fichiers GeoJSON"""
##
gdf = gpd.GeoDataFrame(
    df_arboles.dropna(subset=["geometry"]),
    geometry="geometry",
    crs="EPSG:4326"
)

##transforme les coordonnées CRS à coordonnées métricas
# Proyectar a coordenadas métricas
gdf_m = gdf.to_crs("EPSG:2154")

## Je fixe un point de repère sur le MAP
# Punto de referencia
punto_totem = gpd.GeoSeries(
    [gpd.points_from_xy([lon_totem], [lat_totem])[0]], #Référence campus numérique Grenoble
    crs="EPSG:4326"
).to_crs("EPSG:2154").iloc[0]


## Calcule la distance d'une série de points par rappor à un point donné
# Calcular distancias en metros
gdf_m["distancia_m"] = gdf_m.geometry.distance(punto_totem)


# Obtener los 10 árboles más cercanos
top10 = (
    gdf_m.sort_values("distancia_m") # Je trie l'object selon la colonne de distance en mètres
    .head(10)
    [[
        "ELEM_POINT_ID",
        "NOM",
        "GENRE_BOTA",
        "ESPECE",
        "distancia_m"
    ]]
    .copy()
)

## Renvoie number arrondi avec une précision de ndigits
#J'arrondis à 2.
top10["distancia_m"] = top10["distancia_m"].round(2)

print(top10.to_string(index=False))

print('Mapa de representacion')

##
### Code pour l'impresion en map
## Fontion qui permet de générer una carte centrée sue le point de référence donné 
## Et d'afficher les 10 arbres les plus proches de ce point
## reçoit la latitude, la longitude et la Geometry contenues dans le DataFrame
def generer_map(lat_ref, lon_ref, geodf):
    # Je fixe le map sur le point de référence 
    mapa = folium.Map(
        location=[lat_ref, lon_ref],
        zoom_start=16
    )
    
    # Marcar Le TOTEM (point de référence donnée)
    folium.Marker(
        [lat_ref, lon_ref],
        popup="Le TOTEM",
        icon=folium.Icon(color="red", icon="home")
    ).add_to(mapa)

    # Marcar los 10 árboles
    # geodf ====>>>  Geometrie DataFrame
    for _, row in geodf.loc[top10.index].iterrows():
        punto = row.geometry
        lon, lat = gpd.GeoSeries(
            [punto], crs="EPSG:2154"
        ).to_crs("EPSG:4326").iloc[0].coords[0]
    
        folium.CircleMarker(
            location=[lat, lon],
            radius=6,
            popup=(
                f"Árbol ID: {row['ELEM_POINT_ID']}<br>"
                f"Especie: {row['ESPECE']}<br>"
                f"Distancia: {row['distancia_m']:.2f} m"
            ),
            color="green",
            fill=True
        ).add_to(mapa)

    return mapa
generer_map(lat_totem, lon_totem, gdf_m)


 ELEM_POINT_ID      NOM   GENRE_BOTA     ESPECE  distancia_m
         26633 ESP11973       Ginkgo     biloba        28.48
          4340 ESP27119       Prunus  serrulata        37.32
         11941 ESP28209       Ginkgo     biloba        39.42
         19236  ESP5523       Prunus  serrulata        40.52
          4339 ESP16463    Tetradium   danielli        41.18
         26631  ESP8829       Prunus  serrulata        42.90
         15599 ESP27671       Ginkgo     biloba        44.79
         26634 ESP22275 Koelreuteria paniculata        45.17
         22952 ESP26412    Tetradium   danielli        45.58
         15598 ESP22806 Koelreuteria paniculata        45.81
Mapa de representacion


<h1> Quelle est la distance moyenne entre les arbres ? </h1>
<h2> Pour cet exercice, je prendrai les 10 derniers arbres de la base de données!</h2>

In [23]:
## J'appelle la fonction de chargement pour créer una 
## Object de type DateFrame contenant les informations des arbres de Grenoble
data_arbres_moyenne = charger_data()


###
## Convertir GeoJSON en geometrías
def convertir_geometry(data_arbres_proches):
    df_arboles = data_arbres_proches.copy()
    
    df_arboles["geometry"] = df_arboles["GeoJSON"].apply(
        lambda x: shape(json.loads(x)) if pd.notna(x) else None
    )
    
    # Crear GeoDataFrame
    """Le code EPSG:4326 correspond au système de coordonnées géographiques WGS 84 (World Geodetic System 1984).
    Caractéristiques principales
    • Type de coordonnées : Latitude et Longitude (2D).
    • Unité de mesure : Degrés décimaux.
    • Étendue : Mondiale (de -180° à 180° en longitude, de -90° à 90° en latitude).
    • Utilisation principale : Référencement natif des récepteurs GPS, stockage de données géographiques brutes et fichiers GeoJSON"""
    ##
    gdf = gpd.GeoDataFrame(
        df_arboles.dropna(subset=["geometry"]),
        geometry="geometry",
        crs="EPSG:4326"
    )
    
    ##transforme les coordonnées CRS à coordonnées métricas
    # Proyectar a coordenadas métricas
    gdf_m = gdf.to_crs("EPSG:2154")
    # print('========================>>>>',type(gdf_m))
    return gdf_m
###


## Calcule la distance moyenne de 10 arbres de la liste
## Reçoit un objet contenant les coordonnées métriques
# from scipy.spatial.distance import pdist
def calculer_distance_moyenne_10(gdf_m):
    # Récupérer les géométries des 10 arbres
    arbres_top10 = gdf_m.loc[top10.index]
    
    # Extraire les coordonnées métriques (x, y)
    coordonnees = [
        (geom.x, geom.y)
        for geom in arbres_top10.geometry
    ]
    
    # Calculer toutes les distances entre les paires d'arbres
    """La distance Euclidienne (ou métrique euclidienne) mesure 
    la longueur du segment de droite qui relie deux points dans un espace. 
    Calculée à l'aide du théorème de Pythagore, elle représente la 
    « distance à vol d'oiseau » dans des domaines comme les mathématiques, 
    la géométrie ou le machine learning"""
    distances = pdist(coordonnees, metric="euclidean") #Instancio la Clase pdist de scipy.spatial.distance
    
    # Distance moyenne en mètres
    distance_moyenne = distances.mean()
    
    print(f"\033[1;31mDistance moyenne entre les 10 arbres :\033[0m \033[1;32m{distance_moyenne:.2f} m \033[0m")
    # return distance_moyenne




# from scipy.spatial import cKDTree
# import numpy as np
def calculer_distance_moyenne(gdf_m):
    # Coordonnées de tous les arbres, en mètres
    """Prenez une séquence de tableaux unidimensionnels et empilez-les en colonnes 
    pour former un tableau bidimensionnel. Les tableaux bidimensionnels sont empilés 
    tels quels, comme pour les tableaux unidimensionnels hstack. Ces derniers sont 
    d'abord transformés en colonnes bidimensionnelles"""
    coordonnees = np.column_stack([
        gdf_m.geometry.x,
        gdf_m.geometry.y
    ])
    
    # Construire un arbre spatial
    """Cette classe fournit un index dans un ensemble de points 
    à k dimensions qui peut être utilisé pour rechercher 
    rapidement les voisins les plus proches de n'importe quel point."""
    tree = cKDTree(coordonnees)
    
    # k=2 : arbre lui-même (distance 0) + voisin le plus proche
    distances, indices = tree.query(coordonnees, k=2)
    
    # Distance au plus proche voisin
    gdf_m["distance_voisin_m"] = distances[:, 1]
    
    # Moyenne
    moyenne = gdf_m["distance_voisin_m"].mean()
    
    print(f"\033[1;31mDistance moyenne au plus proche arbre : \033[1;32m{moyenne:.2f} m \033[0m")
    # return moyenne
###

data_geometry = convertir_geometry(data_arbres_moyenne)
calculer_distance_moyenne_10(data_geometry)
print(' ')
calculer_distance_moyenne(data_geometry)

Distance moyenne entre les 10 arbres : 24.39 m 
 
Distance moyenne au plus proche arbre : 6.17 m 


<h1>Quel est l'arbre le plus entouré</h1>

# 

In [67]:
## J'appelle la fonction de chargement pour créer una 
## Object de type DateFrame contenant les informations des arbres de Grenoble
data_arbres = charger_data()

# =========================================================
# 1. CONFIGURACIÓN
# =========================================================

RADIO_METROS = 5
K_VECINOS = 5  # Número de vecinos usados para la distancia media

###
## Convertir GeoJSON en geometrías
def convertir_geometry(data_arbres_proches):
    df_arboles = data_arbres_proches.copy()
    
    df_arboles["geometry"] = df_arboles["GeoJSON"].apply(
        lambda x: shape(json.loads(x)) if pd.notna(x) else None
    )
    
    # Crear GeoDataFrame
    """Le code EPSG:4326 correspond au système de coordonnées géographiques WGS 84 (World Geodetic System 1984).
    Caractéristiques principales
    • Type de coordonnées : Latitude et Longitude (2D).
    • Unité de mesure : Degrés décimaux.
    • Étendue : Mondiale (de -180° à 180° en longitude, de -90° à 90° en latitude).
    • Utilisation principale : Référencement natif des récepteurs GPS, stockage de données géographiques brutes et fichiers GeoJSON"""
    ##
    gdf = gpd.GeoDataFrame(
        df_arboles.dropna(subset=["geometry"]),
        geometry="geometry",
        crs="EPSG:4326"
    )
    
    ##transforme les coordonnées CRS à coordonnées métricas
    # Proyectar a coordenadas métricas
    gdf_m = gdf.to_crs("EPSG:2154")
    # print('========================>>>>',type(gdf_m))
    return gdf_m
###


gdf_m = convertir_geometry(data_arbres)
# =========================================================
# 4. aqui declaro la funcion que recibe un objet detaframe con la geometry (gdf_m)
# =========================================================

coordenadas = np.column_stack([
    gdf_m.geometry.x.to_numpy(),
    gdf_m.geometry.y.to_numpy()
])

# Árbol espacial para consultas de proximidad
arbol_espacial = cKDTree(coordenadas)

n = len(gdf_m)


# =========================================================
# 5. CRITERIO 1: NÚMERO DE ÁRBOLES EN UN RADIO DE 5 M
# =========================================================

indices_radio = arbol_espacial.query_ball_point(
    coordenadas,
    r=RADIO_METROS
)

# Excluir el propio árbol, pero mantener otros árboles
# que estén exactamente en las mismas coordenadas.
vecinos_5m = [
    [j for j in indices if j != i]
    for i, indices in enumerate(indices_radio)
]

gdf_m["arboles_en_5m"] = [
    len(vecinos) for vecinos in vecinos_5m
]


# =========================================================
# 6. CRITERIO 2: DISTANCIA MEDIA A LOS 5 VECINOS MÁS CERCANOS
# =========================================================

k = min(K_VECINOS, n - 1)

# Consultamos k+1 posiciones para incluir el propio árbol.
distancias, indices = arbol_espacial.query(
    coordenadas,
    k=k + 1
)

medias = []

for i in range(n):
    # Eliminar el índice del propio árbol, no simplemente
    # la distancia cero, para manejar puntos coincidentes.
    distancias_vecinos = distancias[i][indices[i] != i][:k]

    if len(distancias_vecinos) == 0:
        medias.append(np.nan)
    else:
        medias.append(distancias_vecinos.mean()) #Calculo el promedio(loyenne)

gdf_m["distancia_media_vecinos_m"] = medias


# =========================================================
# 7. IDENTIFICAR LOS GANADORES
# =========================================================

# Ganador del criterio 1:
# mayor número de árboles dentro de 5 metros.
max_vecinos = gdf_m["arboles_en_5m"].max()

ganadores_radio = gdf_m.loc[
    gdf_m["arboles_en_5m"] == max_vecinos
].copy()
#print('#############====================================================>>>>>>>>>>>>>',ganadores_radio)
# Ganador del criterio 2:
# menor distancia media a los k vecinos más cercanos.
min_distancia = gdf_m["distancia_media_vecinos_m"].min()

ganadores_distancia = gdf_m.loc[
    np.isclose(
        gdf_m["distancia_media_vecinos_m"],
        min_distancia
    )
].copy()


# =========================================================
# 8. MOSTRAR RESULTADOS
# =========================================================

columnas_resultado = [
    "ELEM_POINT_ID",
    "NOM",
    "GENRE_BOTA",
    "ESPECE",
    "arboles_en_5m",
    "distancia_media_vecinos_m"
]

print("\n=== GANADOR: MÁS ÁRBOLES EN 5 METROS ===")
print(
    ganadores_radio[columnas_resultado]
    .to_string(index=False)
)

print("\n=== GANADOR: MENOR DISTANCIA MEDIA ===")
print(
    ganadores_distancia[columnas_resultado]
    .to_string(index=False)
)


# =========================================================
# 9. CLASIFICACIÓN GENERAL
# =========================================================

# Ordenar primero por cantidad de vecinos (descendente)
# y después por distancia media (ascendente).
ranking = gdf_m.sort_values(
    by=[
        "arboles_en_5m",
        "distancia_media_vecinos_m"
    ],
    ascending=[False, True]
).copy()
ranking2 = ranking.copy()
print('XXXXXX++++++++++++++++++++++++++++++++++++++++++++++++++++++++>>>>>',ranking2)
print("\n=== TOP 10 ÁRBOLES MÁS RODEADOS ===")
print(
    ranking[columnas_resultado]
    .head(10)
    .to_string(index=False)
)


# =========================================================
# 10. GUARDAR LOS RESULTADOS
# =========================================================

# Exportar el ranking sin la geometría, en formato CSV
# ranking.drop(columns="geometry").to_csv(
#     "ranking_arboles_rodeados.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

print("\nRanking guardado en ranking_arboles_rodeados.csv")

# arbre_enturne = gdf_m.query('ELEM_POINT_ID == 24581')
# print('La Fila: ',arbre_enturne)
# arbre_enturne = arbre_enturne['GeoJSON']
# print(arbre_enturne)
# print(type(arbre_enturne))
# arbre_enturne = arbre_enturne.to_string()
# print(arbre_enturne[1])


##
### Code pour l'impresion en map
## Fontion qui permet de générer una carte centrée sue le point de référence donné 
## Et d'afficher les 10 arbres les plus proches de ce point
## reçoit la latitude, la longitude et la Geometry contenues dans le DataFrame
def generer_map(lat_ref, lon_ref, ranking2):
    # Je fixe le map sur le point de référence 
    ranking2 = ranking2.head(10)
    print
    mapa = folium.Map(
        location=[lat_ref, lon_ref],
        zoom_start=16
    )
    
    # Marcar Le TOTEM (point de référence donnée)
    folium.Marker(
        [lat_ref, lon_ref],
        popup="Arbres Plus Enture",
        icon=folium.Icon(color="red", icon="home")
    ).add_to(mapa)

    # Marcar los 10 árboles
    # geodf ====>>>  Geometrie DataFrame
    for _, row in ranking2.iterrows():
        punto = row.geometry
        lon, lat = gpd.GeoSeries(
            [punto], crs="EPSG:2154"
        ).to_crs("EPSG:4326").iloc[0].coords[0]
    
        folium.CircleMarker(
            location=[lat, lon],
            radius=6,
            popup=(
                f"Árbol ID: {row['ELEM_POINT_ID']}<br>"
                f"Especie: {row['ESPECE']}<br>"
                #"f"Distancia: {row['distancia_m']:.2f} m"
            ),
            color="green",
            fill=True
        ).add_to(mapa)

    return mapa
generer_map(45.1772936732712, 5.72169349587152, ranking2)


=== GANADOR: MÁS ÁRBOLES EN 5 METROS ===
 ELEM_POINT_ID     NOM GENRE_BOTA  ESPECE  arboles_en_5m  distancia_media_vecinos_m
         24581 ESP8304   Carpinus betulus             26                   1.814408

=== GANADOR: MENOR DISTANCIA MEDIA ===
 ELEM_POINT_ID      NOM GENRE_BOTA  ESPECE  arboles_en_5m  distancia_media_vecinos_m
         11135 ESP23925   Carpinus betulus             13                   0.678971
XXXXXX++++++++++++++++++++++++++++++++++++++++++++++++++++++++>>>>>        ELEM_POINT_ID      CODE       NOM GENRE  GENRE_DESC CATEGORIE  \
28095          24581   ESP8304   ESP8304   VEG  VEGETATION     ESP01   
28229           2169   ESP1465   ESP1465   VEG  VEGETATION     ESP01   
30333          12807  ESP22715  ESP22715   VEG  VEGETATION     ESP01   
28111          20871   ESP7176   ESP7176   VEG  VEGETATION     ESP01   
27988           2171  ESP10735  ESP10735   VEG  VEGETATION     ESP01   
...              ...       ...       ...   ...         ...       ...   
17291   